In [0]:
import os
import re
import pandas as pd

In [0]:
#dbutils.widgets.remove("Campaign_name")

In [0]:
UPC_DIR = "abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/upc_list"
 
dbutils.widgets.text("Campaign_name", "enter_campaign_here", "Campaign name")
campaign_name = dbutils.widgets.get("Campaign_name").strip()

In [0]:
RAW_UPCS = """
    "0000000000000", "0001111001396", "0001111002161", "0001111002165", "0001111002259", "0001111002679",
    "0001111002680", "0001111003023", "0001111003174", "0001111003179", "0001111003181", "0001111003189",
    "0001111003192", "0001111003195", "0001111003196", "0001111003209", "0001111003392", "0001111004259",
    "0001111004296", "0001111005263", "0001111005805", "0001111006979", "0001111007571", "0001111008029",
    "0001111008948", "0001111009288", "0001111009295", "0001111010084", "0001111010088", "0001111011095",
    "0001111011096", "0001111015996", "0001111016273", "0001111016274", "0001111016369", "0001111016370",
    "0001111016476", "0001111016501", "0001111016502", "0001111016512", "0001111016542", "0001111016543",
    "0001111016550", "0001111016552", "0001111016570", "0001111016602", "0001111016603", "0001111016604",
    "0001111016615", "0001111016642", "0001111016643", "0001111016644", "0001111016645", "0001111016648",
    "0001111016661", "0001111016662", "0001111016673", "0001111016696", "0001111016699", "0001111016724",
    "0001111016757", "0001111016758", "0001111016769", "0001111016806", "0001111016827", "0001111016863",
    "0001111016864", "0001111016899", "0001111016917", "0001111016918", "0001111016919", "0001111016923",
    "0001111016924", "0001111016958", "0001111016959", "0001111017116", "0001111017117", "0001111017183",
    "0001111044045", "0001111065051", "0001111067780", "0001111069177", "0001111069178", "0001111069179",
    "0001111069351", "0001111069532", "0001111069533", "0001111069540", "0001111069541", "0001111069542",
    "0001111069749", "0001111069750", "0001111069795", "0001111069993", "0001111069994", "0001111070002",
    "0001111070051", "0001111070053", "0001111070056", "0001111070057", "0001111070058", "0001111070095",
    "0001111070104", "0001111070201", "0001111070264", "0001111070265", "0001111070286", "0001111070295",
    "0001111070296", "0001111070297", "0001111070298", "0001111070299", "0001111070419", "0001111070420",
    "0001111070482", "0001111070497", "0001111070507", "0004126002704", "0004126002778", "0004126002779",
    "0004126002780", "0004126002781", "0004126002800", "0021615550000", "0021884000000", "0021884050000",
    "0022368600000", "0023858500000", "0023858520000", "0023858540000", "0023858560000", "0025858520000",
    "0025858540000", "0025858580000",
"""
 

In [0]:
tokens = re.findall(r"\d+", RAW_UPCS)
too_long = [t for t in tokens if len(t) > 13]
if too_long:
    raise ValueError(f"{len(too_long)} value(s) longer than 13 digits, e.g. {too_long[:3]}")
 
promoted_upc_list = list(dict.fromkeys(t.zfill(13) for t in tokens))
print(f"{len(tokens)} pasted, {len(promoted_upc_list)} unique UPCs")

In [0]:
campaign_name = dbutils.widgets.get("Campaign_name").strip()

path = f"{UPC_DIR.rstrip('/')}/{campaign_name}.csv"

dbutils.fs.mkdirs(UPC_DIR)
csv_text = pd.DataFrame({"upc": promoted_upc_list}).to_csv(index=False)
dbutils.fs.put(path, csv_text, overwrite=True)

print(f"Saved to {path}")
print("In the main notebook, set the campaign_name widget to:", campaign_name)

In [0]:
print(dbutils.widgets.getAll())